In [ ]:
#%pip uninstall torch torchvision torchaudio -y

In [ ]:
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import random
import numpy as np
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import copy
from datetime import datetime


def set_random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


set_random_seed(42)

In [2]:
# --- 1. Настройка окружения и девайса ---
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Используем GPU для ускорения.")
elif torch.backends.mps.is_available():
    device = torch.device("mps") # Для Mac M1/M2
    print("Используем MPS (Mac) для ускорения.")
else:
    print("Используем CPU. Это может быть медленно.")


Используем GPU для ускорения.


In [3]:
# ==========================================
# 2. Функция вычисления метрик
# ==========================================

def evaluate_metrics(model, test_loader, loss_fn, device):
    """
    Вычисляет Accuracy, True Positive Rate (TPR) и Optimality Gap (Loss).
    """
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    # Списки для сбора всех предсказаний и истинных меток (нужны для F1)
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)

            # 1. Считаем Loss (Это наш прокси для Optimality Gap)
            loss = loss_fn(output, target)
            total_loss += loss.item() * data.size(0)

            # 2. Считаем Accuracy
            _, pred = torch.max(output.data, 1)
            correct += (pred == target).sum().item()
            total += data.size(0)

            # Сохраняем данные для F1-Score
            all_targets.extend(target.cpu().numpy())
            all_preds.extend(pred.cpu().numpy())

    # Итоговые метрики
    accuracy = 100.0 * correct / total
    avg_loss = total_loss / total

    # Вычисляем Macro F1-Score
    macro_f1 = f1_score(all_targets, all_preds, average='macro') * 100.0

    return accuracy, macro_f1, avg_loss

def apply_test_trigger(images):
    """
    Накладывает триггер на 100% картинок для замера ASR.
    """
    poisoned_images = images.clone()
    
    # Для теста полного DBA или BadNet применяем цельный триггер
    poisoned_images[:, :, 28:31, 28:31] = 2.5 

    return poisoned_images

def evaluate_asr(global_model, test_loader, device, target_class=0):
    """
    Вычисляет Attack Success Rate (ASR).
    Возвращает процент успешных срабатываний бэкдора.
    """
    global_model.eval()
    successful_attacks = 0
    total_backdoor_samples = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            # 1. Отфильтровываем картинки, которые И ТАК принадлежат целевому классу
            # (нам не нужно обманывать модель, чтобы самолет распознался как самолет)
            non_target_mask = target != target_class
            if not non_target_mask.any():
                continue
                
            clean_data = data[non_target_mask]
            
            # 2. Накладываем триггер на все отфильтрованные картинки
            poisoned_data = apply_test_trigger(clean_data)
            poisoned_data = poisoned_data.to(device)
            
            # 3. Делаем предсказание
            outputs = global_model(poisoned_data)
            predictions = outputs.argmax(dim=1)
            
            # 4. Считаем, сколько раз модель выдала target_class (успех хакера)
            successful_attacks += (predictions == target_class).sum().item()
            total_backdoor_samples += len(clean_data)
            
    # Вычисляем процент успеха
    asr = (successful_attacks / total_backdoor_samples) * 100.0 if total_backdoor_samples > 0 else 0.0
    return asr

In [4]:
# ==========================================
# 3.1 АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ (CNN для MNIST)
# ==========================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = x.view(-1, 32 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ==========================================
# 3.2 АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ (ResNet-18 для датасета CIFAR-10)
# ==========================================

def get_resnet18_cifar10():
    """
    Создает модифицированную модель ResNet-18, адаптированную для CIFAR-10.
    """
    # Загружаем архитектуру без весов,
    # заменяем все слои BatchNorm на GroupNorm (группируем по 2 канала)
    model = models.resnet18(
        weights=None,
        norm_layer=lambda channels: nn.GroupNorm(num_groups=2, num_channels=channels)
    )

    # Модификация первого сверточного слоя для CIFAR-10 (32x32)
    # По умолчанию kernel_size=7, stride=2, padding=3 (для Imagenet 224x224)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

    # Модификация финального полносвязного слоя для 10 классов
    # (ResNet-18 перед fc выдает 512 признаков)
    model.fc = nn.Linear(512, 10)

    return model

# ==========================================
# --- Multi-Layer Perceptron (MLP) ---
# ==========================================
class MNIST_MLP(nn.Module):
    def __init__(self):
        super(MNIST_MLP, self).__init__()
        # MNIST: 28*28 = 784 входных признаков
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(-1, 784) # "Выпрямляем" картинку в вектор
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# ==========================================
# --- Классический LeNet-5 ---
# ==========================================
class MNIST_LeNet(nn.Module):
    def __init__(self):
        super(MNIST_LeNet, self).__init__()
        # Вход: 1 канал (серый), выход: 6 фильтров, ядро 5x5
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2) # Padding=2 сохраняет 28x28
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # После conv + pool размер станет 5x5
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [5]:
# ==========================================
# 4. КЛИЕНТСКАЯ ЧАСТЬ (Модуль SAM)
# ==========================================
def compute_sam_gradient(model, loss_fn, data, target, rho=0.05):
    """
    Вычисляет робастный градиент с помощью Sharpness-Aware Minimization.
    """
    # Сохраняем исходные веса w^t, чтобы вернуться к ним после "прыжка"
    original_weights = {name: param.clone() for name, param in model.named_parameters()}

    # --- Шаг 1: Вычисляем обычный градиент g = \nabla F(w^t) ---
    model.zero_grad()
    predictions = model(data)
    loss = loss_fn(predictions, target)
    loss.backward()

    # Вычисляем общую L2-норму градиентов со всех слоев модели
    grad_norm = torch.norm(
        torch.stack([param.grad.norm(p=2) for param in model.parameters() if param.grad is not None]),
        p=2
    )

    # --- Шаг 2: Виртуальный прыжок в худшую точку (w^t + \epsilon) ---
    if grad_norm > 0:
        # scale = \rho / ||g||_2  (добавляем 1e-12 для защиты от деления на ноль)
        scale = rho / (grad_norm + 1e-12)
        for param in model.parameters():
            if param.grad is not None:
                e_w = param.grad * scale
                param.data.add_(e_w) # Физически сдвигаем веса в точку w + \epsilon

    # --- Шаг 3: Вычисление SAM-градиента в этой "плохой" точке ---
    model.zero_grad() # Очищаем старые градиенты
    predictions_adv = model(data)
    loss_adv = loss_fn(predictions_adv, target)
    loss_adv.backward() # Теперь в param.grad лежит g^{SAM}

    # Собираем итоговые SAM-градиенты в список для отправки на сервер
    sam_gradients = []
    for param in model.parameters():
        if param.grad is not None:
            sam_gradients.append(param.grad.clone())

    # ВАЖНО: Возвращаем веса модели обратно в исходную точку w^t
    for name, param in model.named_parameters():
        param.data.copy_(original_weights[name])

    return sam_gradients

In [6]:
# ==========================================
# 5. СЕРВЕРНАЯ ЧАСТЬ (L2-Нормализация и Агрегация)
# ==========================================
def server_aggregate_and_update(global_model, client_gradients_list, eta_glob=1.0):
    """
    Собирает градиенты, жестко нормализует их и обновляет глобальную модель.
    client_gradients_list: список списков градиентов от каждого клиента.
    """
    # Инициализируем пустой аккумулятор для итогового вектора
    aggregated_grad = [torch.zeros_like(param) for param in global_model.parameters()]
    num_clients = len(client_gradients_list)

    # Обрабатываем градиент каждого клиента по очереди
    for client_grads in client_gradients_list:

        # Вычисляем L2-норму вектора, присланного клиентом (даже если он хакер)
        client_norm = torch.norm(
            torch.stack([g.norm(p=2) for g in client_grads]),
            p=2
        )

        # L2-Нормализация и усреднение (добавляем вклад клиента)
        for i, g in enumerate(client_grads):
            # Жестко делаем длину вектора равной 1
            normalized_g = g / (client_norm + 1e-12)
            # Добавляем в общую копилку (с весом 1/M)
            aggregated_grad[i] += normalized_g / num_clients

    # --- Шаг обновления глобальной модели (w^{t+1} = w^t - \eta_{glob} * g_{agg}) ---
    with torch.no_grad():
        for param, avg_grad in zip(global_model.parameters(), aggregated_grad):
            param.data.sub_(eta_glob * avg_grad)

In [7]:
# ==========================================
# 6. Функция распределения Дирихле (Non-IID)
# ==========================================
def partition_data_dirichlet(dataset, num_clients, beta=0.5, num_classes=10):
    """
    Разделяет набор данных между клиентами используя распределение Дирихле.

    Параметры:
        dataset: PyTorch Dataset (например, MNIST или CIFAR-10)
        num_clients: Количество клиентов
        beta: Параметр концентрации Дирихле (меньше = больше перекос/Non-IID)
        num_classes: Количество классов в датасете (для MNIST/CIFAR = 10)

    Возвращает:
        Список объектов Subset, где каждый Subset содержит данные одного клиента.
    """
    # Извлекаем все метки классов из датасета
    # Для torchvision.datasets (как MNIST) метки хранятся в атрибуте targets
    if hasattr(dataset, 'targets'):
        train_labels = np.array(dataset.targets)
    else:
        # Универсальный (но медленный) способ, если атрибут targets недоступен
        train_labels = np.array([target for _, target in dataset])

    min_size = 0
    min_require_size = 10 # Гарантируем, что каждый клиент получит хоть какие-то данные

    # Массив для хранения индексов данных каждого клиента
    client_indices = [[] for _ in range(num_clients)]

    # Мы повторяем распределение, пока не убедимся, что ни один клиент не остался без данных
    while min_size < min_require_size:
        client_indices = [[] for _ in range(num_clients)]

        for k in range(num_classes):
            # Находим все индексы изображений класса k
            idx_k = np.where(train_labels == k)[0]
            np.random.shuffle(idx_k)

            # Генерируем пропорции распределения этого класса между клиентами
            # Математика: p ~ Dir(beta * 1)
            proportions = np.random.dirichlet(np.repeat(beta, num_clients))

            # Чтобы избежать микро-долей, отсекаем слишком маленькие пропорции
            # и нормализуем вектор, чтобы сумма снова равнялась 1
            proportions = np.array([p * (len(idx_j) < len(train_labels) / num_clients)
                                    for p, idx_j in zip(proportions, client_indices)])
            proportions = proportions / proportions.sum()

            # Превращаем пропорции в конкретное количество образцов для каждого клиента
            proportions = (np.cumsum(proportions) * len(idx_k)).astype(int)[:-1]

            # Разделяем индексы класса k между клиентами согласно пропорциям
            idx_k_split = np.split(idx_k, proportions)

            for i in range(num_clients):
                client_indices[i] += idx_k_split[i].tolist()

        # Проверяем, сколько данных получил самый "бедный" клиент
        min_size = min([len(idx_j) for idx_j in client_indices])

    # Превращаем списки индексов в объекты PyTorch Subset
    client_datasets = [Subset(dataset, indices) for indices in client_indices]

    # Небольшой вывод статистики для наглядности
    print(f"--- Распределение Дирихле (beta={beta}) завершено ---")
    for i, indices in enumerate(client_indices):
        print(f"Клиент {i+1}: {len(indices)} образцов данных")

    return client_datasets

In [8]:
# ================================
# 7. В ЭТОМ БЛОКЕ РЕАЛИЗУЮТСЯ АТАКИ
# ================================

# ================================
# 7.1 Атаки на веса (Model / Gradient Poisoning)
# ================================
def apply_gradient_attacks(client_gradients, num_attackers, attack_type="lie", z_val=1.5, c_val=100.0, gamma=5.0):
    """
    Применяет византийские атаки к первым `num_attackers` клиентам. Учитывая случайность распределения
    данных между клиентами, это не нарушает обобщенность атаки.
    client_gradients: список градиентов от всех клиентов.
    """
    if num_attackers == 0:
        return client_gradients
        
    num_clients = len(client_gradients)
    honest_gradients = client_gradients[num_attackers:]
    attacked_gradients = client_gradients.copy()
    
    # Собираем честные градиенты для вычисления статистики (нужно для умных атак)
    # Формат: список тензоров для каждого слоя
    honest_stacked = [torch.stack([h[layer_idx] for h in honest_gradients]) 
                      for layer_idx in range(len(honest_gradients[0]))]

    for i in range(num_attackers):
        malicious_grad = []
        for layer_idx in range(len(client_gradients[0])):
            honest_layer_stack = honest_stacked[layer_idx]

            # Gaussian attack (Fed-NGA)
            if attack_type == "gaussian":
                # Генерируем шум с теми же средним и дисперсией, что у честных
                mean = torch.mean(honest_layer_stack, dim=0)
                std = torch.std(honest_layer_stack, dim=0) + 1e-6
                noise = torch.randn_like(mean) * std + mean
                malicious_grad.append(noise)
                
            # Same-value attack (Fed-NGA)
            elif attack_type == "same_value":
                # Заполняем весь тензор константой (по умолчанию 100.0)
                malicious_grad.append(torch.full_like(honest_layer_stack[0], c_val))
                
            # Double Attack (FedLAW)
            elif attack_type == "double":
                # Умножаем честный градиент самого хакера на -2 (Sign-Flipping x2)
                malicious_grad.append(-gamma * client_gradients[i][layer_idx])
                
            # Lie Attack / ALIE - A Little Is Enough (FedLAW)
            elif attack_type == "lie":
                # A Little Is Enough (ALIE)
                mean = torch.mean(honest_layer_stack, dim=0)
                std = torch.std(honest_layer_stack, dim=0) + 1e-6
                # Смещаемся от среднего на z_val стандартных отклонений
                stealth_vector = mean - z_val * std
                malicious_grad.append(stealth_vector)
                
        attacked_gradients[i] = malicious_grad
        
    return attacked_gradients

# ================================
# 7.2 Атаки через бэкдоры (Data Poisoning)
# ================================
# Сначала создадим вспомогательную функцию, которая накладывает триггер на тензоры изображений прямо в процессе итерации батча.
def poison_batch(images, targets, attack_type="badnet", target_class=0, poison_ratio=0.5):
    """
    Накладывает триггер на часть батча.
    poison_ratio: какой процент картинок в батче будет отравлен (0.5 = 50%).
    """
    poisoned_images = images.clone()
    poisoned_targets = targets.clone()
    
    num_to_poison = int(len(images) * poison_ratio)
    if num_to_poison == 0:
        return poisoned_images, poisoned_targets

    # Выбираем случайные индексы в батче для отравления
    indices = torch.randperm(len(images))[:num_to_poison]
    
    for i in indices:
        if attack_type == "badnet":
            # Стандартный триггер: белый квадрат 3x3 в углу
            poisoned_images[i, :, 28:31, 28:31] = 2.5 
            
        elif attack_type == "dba_part1":
            # Только верхний левый пиксель триггера
            poisoned_images[i, :, 28, 28] = 2.5
            
        elif attack_type == "dba_part2":
            # Только верхний правый пиксель
            poisoned_images[i, :, 28, 30] = 2.5
            
        elif attack_type == "dba_part3":
            # Только нижний левый пиксель
            poisoned_images[i, :, 30, 28] = 2.5
            
        elif attack_type == "dba_part4":
            # Только нижний правый пиксель
            poisoned_images[i, :, 30, 30] = 2.5
            
        poisoned_targets[i] = target_class
        # здесь можно добавить ещё всяких приколов
            
    return poisoned_images, poisoned_targets

# Этот метод заменяет стандартный client_update для тех клиентов, которые помечены как «хакеры».
def client_update_malicious(global_model, train_loader, device, eta_loc, attack_type="badnet", scale_factor=1.0):
    """
    Обучение клиента-злоумышленника (Backdoor + Model Replacement).
    scale_factor: для атаки Backdoor Model Replacement из FedLAW (обычно num_clients / eta_glob).
    """
    local_model = copy.deepcopy(global_model).to(device)
    local_model.train()
    optimizer = torch.optim.SGD(local_model.parameters(), lr=eta_loc)
    loss_fn = torch.nn.CrossEntropyLoss()
    
    for data, target in train_loader:
        # ШАГ 1: Травим данные перед подачей в модель
        data, target = poison_batch(data, target, attack_type=attack_type)
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = local_model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        
    # ШАГ 2: Вычисляем псевдо-градиент
    client_pseudo_grad = []
    with torch.no_grad():
        for p_glob, p_loc in zip(global_model.parameters(), local_model.parameters()):
            diff = p_glob.data - p_loc.data
            # Применяем масштабирование (Model Replacement Attack), если нужно
            client_pseudo_grad.append(diff * scale_factor)
            
    return client_pseudo_grad

In [15]:
# ==========================================
# 8.1 ОСНОВНОЙ ЦИКЛ СИМУЛЯЦИИ (Тестирование)
# ==========================================
# Модель CNN для датасета MNIST
# ==========================================

print("Загрузка данных MNIST...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Настройки симуляции
num_clients = 10
rounds = 400
batch_size = 64
num_attackers = 4  # 1 хакер из 8 клиентов
attack_type = "lie" # "gaussian", "same_value", "double", "lie"
attack_types_group2 = ["badnet", "dba", "MRB"]
eta_glob = 0.5

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 0.1 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
set_random_seed(42)
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)

loss_fn = nn.CrossEntropyLoss()


Загрузка данных MNIST...
--- Распределение Дирихле (beta=0.1) завершено ---
Клиент 1: 8208 образцов данных
Клиент 2: 7616 образцов данных
Клиент 3: 6235 образцов данных
Клиент 4: 2042 образцов данных
Клиент 5: 11046 образцов данных
Клиент 6: 4723 образцов данных
Клиент 7: 7348 образцов данных
Клиент 8: 6270 образцов данных
Клиент 9: 385 образцов данных
Клиент 10: 6127 образцов данных


C:\Users\Алексей\AppData\Local\Temp\ipykernel_22700\184332757.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_labels = np.array(dataset.targets)


In [16]:
# Атаки из группы 1
if attack_type == "gaussian":
    print("Атака: Gaussian attack")
elif attack_type == "same_value":
    print("Атака: Same-value attack")
elif attack_type == "double":
    print("Атака: Double Attack")
elif attack_type == "lie":
    print("Атака: Lie Attack / ALIE - A Little Is Enough")

print(f"Количество атакующих: {num_attackers}\n==========================================================================================")
print(f"Запуск Fed-SANA: Клиентов={num_clients}, Раундов={rounds}")
start_time = datetime.now()

# Инициируем модель
global_model = MNIST_LeNet().to(device)

accuracy_list, f1_list, optimality_gap_list = [], [], []  # Создаем списки для метрик

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # Создаем пустые тензоры для накопления градиентов за всю эпоху
        client_accumulated_grads = [torch.zeros_like(param) for param in global_model.parameters()]
        num_batches = 0

        # === Проходим по ВСЕМ батчам клиента (Локальная Эпоха) ===
        for data, target in client_loaders[client_id]:
            data, target = data.to(device), target.to(device)

            # Вычисляем SAM-градиент для текущего батча (64 картинки)
            batch_grads = compute_sam_gradient(global_model, loss_fn, data, target, rho=0.05)

            # Прибавляем градиенты батча к общей сумме
            for i, g in enumerate(batch_grads):
                client_accumulated_grads[i] += g

            num_batches += 1

        # Усредняем градиенты за всю эпоху
        client_avg_grads = [g / num_batches for g in client_accumulated_grads]
        client_gradients.append(client_avg_grads)

    # === Внедрение византийской атаки ===
    client_gradients = apply_gradient_attacks(client_gradients, num_attackers, attack_type=attack_type)

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты и делает шаг
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)
    accuracy_list.append(accuracy)
    f1_list.append(f1)
    optimality_gap_list.append(optimality_gap)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {str(datetime.now()-start_time).split('.')[0]}")

    global_model.train() # Возвращаем модель в режим обучения

print(f"Accuracy max: {max(accuracy_list):.2f}% | F1-Score max: {max(f1_list):.2f}% | Opt. Gap min: {min(optimality_gap_list):.4f}")


Атака: Lie Attack / ALIE - A Little Is Enough
Количество атакующих: 4
Запуск Fed-SANA: Клиентов=10, Раундов=400
Раунд 001/400 | Accuracy: 9.80% | F1-Score: 1.79% | Opt. Gap: 2.3095 | Time: 0:00:11
Раунд 003/400 | Accuracy: 9.84% | F1-Score: 1.86% | Opt. Gap: 2.6665 | Time: 0:00:33
Раунд 005/400 | Accuracy: 16.40% | F1-Score: 5.45% | Opt. Gap: 2.8441 | Time: 0:00:58
Раунд 007/400 | Accuracy: 20.77% | F1-Score: 11.05% | Opt. Gap: 2.8552 | Time: 0:01:21
Раунд 009/400 | Accuracy: 13.58% | F1-Score: 5.01% | Opt. Gap: 2.7652 | Time: 0:01:45
Раунд 011/400 | Accuracy: 9.82% | F1-Score: 1.84% | Opt. Gap: 2.8813 | Time: 0:02:08
Раунд 013/400 | Accuracy: 22.89% | F1-Score: 14.64% | Opt. Gap: 2.7669 | Time: 0:02:31
Раунд 015/400 | Accuracy: 33.54% | F1-Score: 21.71% | Opt. Gap: 2.8844 | Time: 0:02:54
Раунд 017/400 | Accuracy: 36.08% | F1-Score: 23.45% | Opt. Gap: 3.0841 | Time: 0:03:18
Раунд 019/400 | Accuracy: 37.36% | F1-Score: 25.16% | Opt. Gap: 3.2160 | Time: 0:03:41
Раунд 021/400 | Accuracy: 

In [ ]:
# Атаки из группы 2
for attack_type in attack_types_group2: 
    if attack_type == "badnet":
        print("Атака: badnet")
    elif attack_type == "dba":
        print("Атака: dba")
    elif attack_type == "MRB":
        print("Атака: MRB")
    
    for num_attackers in num_attackers_mass:
        print(f"Количество атакующих: {num_attackers}\n==========================================================================================")

        for round_idx in range(rounds):
            client_gradients = []

            # 1. Фаза клиентов
            for client_id in range(num_clients):
                # Если ID клиента меньше num_attackers, он хакер
                if client_id < num_attackers:
                    # Вызываем "отравленное" обучение
                    if attack_type == "MRB":
                        p_grad = client_update_malicious(global_model, client_loaders[client_id], device, eta_glob, attack_type="badnet", scale_factor=100.0)
                    elif attack_type == "badnet":
                        p_grad = client_update_malicious(global_model, client_loaders[client_id], device, eta_glob, attack_type=attack_type, scale_factor=1.0)
                    elif attack_type == "dba":
                        p_grad = client_update_malicious(global_model, client_loaders[client_id], device, eta_glob, attack_type=f"dba_part{client_id+1}", scale_factor=1.0)
                    client_gradients.append(p_grad)
                else:
                    # Создаем пустые тензоры для накопления градиентов за всю эпоху
                    client_accumulated_grads = [torch.zeros_like(param) for param in global_model.parameters()]
                    num_batches = 0

                    # === Проходим по ВСЕМ батчам клиента (Локальная Эпоха) ===
                    for data, target in client_loaders[client_id]:
                        data, target = data.to(device), target.to(device)

                        # Вычисляем SAM-градиент для текущего батча (64 картинки)
                        batch_grads = compute_sam_gradient(global_model, loss_fn, data, target, rho=0.05)

                        # Прибавляем градиенты батча к общей сумме
                        for i, g in enumerate(batch_grads):
                            client_accumulated_grads[i] += g

                        num_batches += 1

                    # Усредняем градиенты за всю эпоху
                    client_avg_grads = [g / num_batches for g in client_accumulated_grads]
                    client_gradients.append(client_avg_grads)

            # 2. Фаза сервера
            # Сервер агрегирует нормализованные градиенты и делает шаг
            server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

            # 3. Оценка глобальной модели (каждые 2 раунда)
            if round_idx % 2 == 0 or round_idx == rounds - 1:
                accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

                # Замеряем ASR на отравленных данных
                current_asr = evaluate_asr(global_model, test_loader, device, target_class=0)

                print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Attack Success Rate: {current_asr:.2f}% | Time: {str(datetime.now()-start_time).split('.')[0]}")

                global_model.train() # Возвращаем модель в режим обучения

In [29]:
# ==========================================
# 8.2 ОСНОВНОЙ ЦИКЛ СИМУЛЯЦИИ (Тестирование)
# ==========================================
# Модель ResNet-18 для датасета CIFAR-10
# ==========================================

print("Загрузка данных CIFAR-10...")
# Трансформации для CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов 

eta_loc = 0.1  # Стандартный шаг для локального обучения ResNet
eta_glob = 1.0  # Шаг сервера

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 0.1 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициализация ResNet-18 для CIFAR-10 на GPU
global_model = get_resnet18_cifar10().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA (ResNet-18 на CIFAR-10): Клиентов={num_clients}, Раундов={rounds}")
start_time = datetime.now()

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # === ВАЖНО: Создаем локальную копию модели для обучения ===
        local_model = copy.deepcopy(global_model).to(device)
        # Если deepcopy работает медленно, используй:
        # local_model = get_resnet18_cifar10().to(device)
        # local_model.load_state_dict(global_model.state_dict())

        # Клиент проходит по всем своим батчам (Локальная Эпоха)
        for data, target in client_loaders[client_id]:
            data, target = data.to(device), target.to(device)

            # Вычисляем SAM-градиент на ЛОКАЛЬНОЙ модели
            batch_grads = compute_sam_gradient(local_model, loss_fn, data, target, rho=0.05)

            # === ЛОКАЛЬНЫЙ ШАГ ОБУЧЕНИЯ (Local SGD) ===
            with torch.no_grad():
                for param, g in zip(local_model.parameters(), batch_grads):
                    param.data.sub_(eta_loc * g)

        # === ВЫЧИСЛЕНИЕ ПСЕВДО-ГРАДИЕНТА ДЛЯ СЕРВЕРА ===
        # Вектор смещения = Старые глобальные веса - Новые локальные веса
        client_pseudo_grad = []
        with torch.no_grad():
            for param_glob, param_loc in zip(global_model.parameters(), local_model.parameters()):
                client_pseudo_grad.append(param_glob.data - param_loc.data)

        client_gradients.append(client_pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {str(datetime.now()-start_time).split('.')[0]}")

        global_model.train() # Возвращаем модель в режим обучения

Загрузка данных CIFAR-10...


c:\Users\Алексей\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


--- Распределение Дирихле (beta=0.1) завершено ---
Клиент 1: 1697 образцов данных
Клиент 2: 5366 образцов данных
Клиент 3: 8952 образцов данных
Клиент 4: 6375 образцов данных
Клиент 5: 5351 образцов данных
Клиент 6: 10218 образцов данных
Клиент 7: 7603 образцов данных
Клиент 8: 4438 образцов данных
Запуск Fed-SANA (ResNet-18 на CIFAR-10): Клиентов=8, Раундов=100
Раунд 001/100 | Accuracy: 10.17% | F1-Score: 3.48% | Opt. Gap: 2.4453 | Time: 0:00:24
Раунд 003/100 | Accuracy: 10.00% | F1-Score: 1.82% | Opt. Gap: 2.3720 | Time: 0:01:40
Раунд 005/100 | Accuracy: 10.00% | F1-Score: 1.82% | Opt. Gap: 2.3914 | Time: 0:03:02
Раунд 007/100 | Accuracy: 17.14% | F1-Score: 5.81% | Opt. Gap: 2.3058 | Time: 0:04:23
Раунд 009/100 | Accuracy: 18.40% | F1-Score: 6.27% | Opt. Gap: 2.2415 | Time: 0:05:45
Раунд 011/100 | Accuracy: 18.72% | F1-Score: 6.44% | Opt. Gap: 2.2172 | Time: 0:07:07
Раунд 013/100 | Accuracy: 18.91% | F1-Score: 6.65% | Opt. Gap: 2.1850 | Time: 0:08:30
Раунд 015/100 | Accuracy: 20.46% 

KeyboardInterrupt: 

In [ ]:
# Добавить вычисление метрики $$I = A_{IID} - A_{Non-IID}$$Где:
#$A_{IID}$ — точность модели при идеально равномерном распределении данных (когда у всех клиентов поровну всех цифр/классов).
#$A_{Non-IID}$ — точность в текущем тесте (например, при Дирихле $\beta=0.5$).
# Надо дважды прогнать модель и посмотреть на пункт Accuracy

# Здесь начинается тестирование предшественников, а именно FedISM, pFedSAM, Fed-NGA, FedLAW

# 9.1 Реализация FedISM

In [9]:
#========================================================
# Функции имитирующие клиентов и сервер
#========================================================
def client_update_fedism(global_model, train_loader, device, eta_loc, rho=0.05):
    """
    Шаг 1 (SALT): Локальное обучение клиента в стиле FedISM.
    Возвращает вектор смещения (псевдо-градиент) и усредненный показатель Sharpness.
    """
    local_model = copy.deepcopy(global_model).to(device)
    loss_fn = torch.nn.CrossEntropyLoss()
    
    total_sharpness = 0.0
    num_batches = 0
    
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        
        # --- Прямой проход для исходных весов (theta) ---
        outputs = local_model(data)
        loss = loss_fn(outputs, target)
        loss.backward()
        loss_theta = loss.item() # Сохраняем L(theta)
        
        # --- Вычисляем возмущение epsilon ---
        with torch.no_grad():
            # Норма градиентов по всем слоям
            grad_norm = torch.norm(torch.stack([p.grad.norm(p=2) for p in local_model.parameters() if p.grad is not None]), p=2)
            scale = rho / (grad_norm + 1e-12)
            
            # Сохраняем epsilon и временно сдвигаем веса в точку (theta + epsilon)
            for p in local_model.parameters():
                if p.grad is not None:
                    p.eps = (p.grad * scale).clone()
                    p.data.add_(p.eps)
                    
        local_model.zero_grad() # Очищаем старые градиенты
        
        # --- Прямой проход для возмущенных весов (theta + epsilon) ---
        outputs_eps = local_model(data)
        loss_eps = loss_fn(outputs_eps, target)
        loss_eps.backward()
        
        # Вычисляем Sharpness батча
        batch_sharpness = loss_eps.item() - loss_theta
        total_sharpness += batch_sharpness
        num_batches += 1
        
        # --- Возвращаем веса и делаем локальный шаг Local SGD ---
        with torch.no_grad():
            for p in local_model.parameters():
                if p.grad is not None:
                    p.data.sub_(p.eps)            # Возвращаемся обратно в theta
                    p.data.sub_(eta_loc * p.grad) # Делаем шаг против градиента, вычисленного в (theta + eps)
                    
    # Усредняем Sharpness за эпоху
    avg_sharpness = total_sharpness / num_batches
    
    # Вычисляем псевдо-градиент (разницу весов для отправки на сервер)
    client_pseudo_grad = []
    with torch.no_grad():
        for param_glob, param_loc in zip(global_model.parameters(), local_model.parameters()):
            client_pseudo_grad.append(param_glob.data - param_loc.data)
            
    return client_pseudo_grad, avg_sharpness


def server_aggregate_fedism(global_model, client_gradients, client_sharpness_list, eta_glob, q=1.0):
    """
    Шаг 2 (SAGA): Глобальная агрегация FedISM.
    Взвешивает обновления клиентов пропорционально их Sharpness.
    """
    # Преобразуем показатели Sharpness всех клиентов в тензор
    sharpness_tensor = torch.tensor(client_sharpness_list, dtype=torch.float32)
    
    # Вычисляем динамические веса через Softmax
    # Чем выше параметр q, тем агрессивнее сервер спасает отстающих клиентов
    weights = F.softmax(q * sharpness_tensor, dim=0)
    
    with torch.no_grad():
        for param_idx, param in enumerate(global_model.parameters()):
            # Создаем пустой тензор для взвешенной суммы
            aggregated_update = torch.zeros_like(param.data)
            
            # Суммируем псевдо-градиенты с учетом их индивидуального веса
            for client_idx, pseudo_grad in enumerate(client_gradients):
                aggregated_update.add_(weights[client_idx] * pseudo_grad[param_idx])
            
            # Сдвигаем глобальную модель
            param.data.sub_(eta_glob * aggregated_update)

In [10]:
# ==========================================
# 9.1 ОСНОВНОЙ ЦИКЛ СИМУЛЯЦИИ (Тестирование)
# ==========================================
# 9.1.1 Реализация алгоритма FedISM.
# Модель LeNet для датасета MNIST
#===========================================

print("Загрузка данных MNIST...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Настройки симуляции
num_clients = 10
rounds = 400
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов
attack_type = "gaussian" # "gaussian", "same_value", "double", "lie"
attack_types_group2 = ["badnet", "dba", "MRB"]
eta_loc = 0.1
eta_glob = 0.5

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 0.1 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
set_random_seed(42)
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)

loss_fn = nn.CrossEntropyLoss()

Загрузка данных MNIST...
--- Распределение Дирихле (beta=0.1) завершено ---
Клиент 1: 8208 образцов данных
Клиент 2: 7616 образцов данных
Клиент 3: 6235 образцов данных
Клиент 4: 2042 образцов данных
Клиент 5: 11046 образцов данных
Клиент 6: 4723 образцов данных
Клиент 7: 7348 образцов данных
Клиент 8: 6270 образцов данных
Клиент 9: 385 образцов данных
Клиент 10: 6127 образцов данных


C:\Users\Алексей\AppData\Local\Temp\ipykernel_19940\184332757.py:20: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  train_labels = np.array(dataset.targets)


In [ ]:
# Атаки из группы 1
if attack_type == "gaussian":
    print("Атака: Gaussian attack")
elif attack_type == "same_value":
    print("Атака: Same-value attack")
elif attack_type == "double":
    print("Атака: Double Attack")
elif attack_type == "lie":
    print("Атака: Lie Attack / ALIE - A Little Is Enough")

print(f"Количество атакующих: {num_attackers}\n==========================================================================================")
print(f"Запуск FedISM: Клиентов={num_clients}, Раундов={rounds}")
start_time = datetime.now()

# Инициируем модель
global_model = MNIST_LeNet().to(device)

accuracy_list, f1_list, optimality_gap_list = [], [], []  # Создаем списки для метрик

for round_idx in range(rounds):
    client_gradients = []
    client_sharpness_list = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        pseudo_grad, sharpness = client_update_fedism(global_model, client_loaders[client_id], device, eta_loc=eta_loc)
        client_gradients.append(pseudo_grad)
        client_sharpness_list.append(sharpness)

    # === Внедрение византийской атаки ===
    client_gradients = apply_gradient_attacks(client_gradients, num_attackers, attack_type=attack_type)

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты и делает шаг
    server_aggregate_fedism(global_model, client_gradients, client_sharpness_list, eta_glob=eta_glob, q=1.0)  # q --- Параметр честности (можно тюнить от 0.5 до 5.0)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)
    accuracy_list.append(accuracy)
    f1_list.append(f1)
    optimality_gap_list.append(optimality_gap)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {str(datetime.now()-start_time).split('.')[0]}")

    global_model.train() # Возвращаем модель в режим обучения

print(f"Accuracy max: {max(accuracy_list):.2f}% | F1-Score max: {max(f1_list):.2f}% | Opt. Gap min: {min(optimality_gap_list):.4f}")

Атака: Gaussian attack
Количество атакующих: 1
Запуск Fed-SANA: Клиентов=10, Раундов=400
Раунд 001/400 | Accuracy: 9.93% | F1-Score: 2.04% | Opt. Gap: 2.2533 | Time: 0:00:11


KeyboardInterrupt: 

In [ ]:
#========================================================
# 9.1.1 Реализация алгоритма FedISM.
# Модель CNN для датасета MNIST
#========================================================

# Инициируем нашу модель
global_model = SimpleCNN().to(device)

print(f"Запуск Fed-SANA: Клиентов={num_clients}, Раундов={rounds}")

for round_idx in range(rounds):
    client_gradients = []
    client_sharpness_list = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        pseudo_grad, sharpness = client_update_fedism(global_model, client_loaders[client_id], device, eta_loc=eta_loc)
        client_gradients.append(pseudo_grad)
        client_sharpness_list.append(sharpness)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты и делает шаг
    server_aggregate_fedism(global_model, client_gradients, client_sharpness_list, eta_glob=eta_glob, q=1.0)  # q --- Параметр честности (можно тюнить от 0.5 до 5.0)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

In [ ]:
#========================================================
# 9.1.2 Реализация алгоритма FedISM.
# Модель ResNet-18 для датасета CIFAR-10
#========================================================
print("Загрузка данных CIFAR-10...")
# Трансформации для CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов 

eta_loc = 0.1  # Стандартный шаг для локального обучения ResNet
eta_glob = 1.0  # Шаг сервера

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициализация ResNet-18 для CIFAR-10 на GPU
global_model = get_resnet18_cifar10().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA (ResNet-18 на CIFAR-10): Клиентов={num_clients}, Раундов={rounds}")

for round_idx in range(rounds):
    client_gradients = []
    client_sharpness_list = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        pseudo_grad, sharpness = client_update_fedism(global_model, client_loaders[client_id], device, eta_loc=eta_loc)
        client_gradients.append(pseudo_grad)
        client_sharpness_list.append(sharpness)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты
    server_aggregate_fedism(global_model, client_gradients, client_sharpness_list, eta_glob=eta_glob, q=1.0)  # q --- Параметр честности (можно тюнить от 0.5 до 5.0)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

# 9.2 Реализация pFedSAM

In [ ]:
def client_update_pfedsam(global_model, local_model, train_loader, device, eta_loc, lam=15.0, rho=0.05):
    """
    Локальное обновление pFedSAM.
    lam: коэффициент персонализации (чем выше, тем ближе к глобальной модели).
    local_model: персонализированная модель клиента с прошлого раунда.
    """
    # Мы обучаем именно локальную модель клиента
    local_model.train()
    optimizer = torch.optim.SGD(local_model.parameters(), lr=eta_loc)
    loss_fn = torch.nn.CrossEntropyLoss()
    
    # Копируем параметры глобальной модели для расчета "штрафа" (proximal term)
    global_params = [p.data.clone().detach() for p in global_model.parameters()]
    
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        
        # --- Шаг 1: Поиск возмущения epsilon (SAM) ---
        output = local_model(data)
        # Ошибка = CrossEntropy + (lam/2) * ||local - global||^2
        prox_loss = 0
        for p, g_p in zip(local_model.parameters(), global_params):
            prox_loss += (lam / 2) * torch.norm(p - g_p)**2
        
        loss = loss_fn(output, target) + prox_loss
        loss.backward()
        
        # Вычисляем epsilon
        with torch.no_grad():
            grad_norm = torch.norm(torch.stack([p.grad.norm(p=2) for p in local_model.parameters() if p.grad is not None]), p=2)
            scale = rho / (grad_norm + 1e-12)
            for p in local_model.parameters():
                if p.grad is not None:
                    p.data.add_(p.grad * scale)
                    p.old_grad = p.grad.clone() # Сохраняем градиент для второго шага
        
        local_model.zero_grad()
        
        # --- Шаг 2: Финальный шаг оптимизации в возмущенной точке ---
        output_adv = local_model(data)
        prox_loss_adv = 0
        for p, g_p in zip(local_model.parameters(), global_params):
            prox_loss_adv += (lam / 2) * torch.norm(p - g_p)**2
            
        loss_adv = loss_fn(output_adv, target) + prox_loss_adv
        loss_adv.backward()
        
        # Возвращаемся из точки theta+eps и делаем шаг
        with torch.no_grad():
            for p in local_model.parameters():
                if p.grad is not None:
                    # Убираем возмущение epsilon
                    grad_norm = torch.norm(torch.stack([p_alt.old_grad.norm(p=2) for p_alt in local_model.parameters() if hasattr(p_alt, 'old_grad')]), p=2)
                    p.data.sub_(p.old_grad * (rho / (grad_norm + 1e-12)))
                    # Делаем реальный шаг обучения
                    p.data.sub_(eta_loc * p.grad)
        
        local_model.zero_grad()

    # Возвращаем псевдо-градиент для обновления ГЛОБАЛЬНОЙ модели
    client_pseudo_grad = []
    with torch.no_grad():
        for p_glob, p_loc in zip(global_model.parameters(), local_model.parameters()):
            client_pseudo_grad.append(p_glob.data - p_loc.data)
            
    return client_pseudo_grad, local_model

In [ ]:
#========================================================
# 9.2.1 Реализация алгоритма pFedSAM.
# Модель CNN для датасета MNIST
#========================================================
print("Загрузка данных MNIST...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов
eta_loc = 0.1
eta_glob = 1.0

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициируем нашу модель
global_model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA: Клиентов={num_clients}, Раундов={rounds}")

# Инициализируем локальные модели для каждого клиента
local_models = [copy.deepcopy(global_model).to(device) for _ in range(num_clients)]

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # Передаем глобальную модель и ПЕРСОНАЛЬНУЮ модель клиента
        pseudo_grad, updated_local = client_update_pfedsam(global_model, local_models[client_id], client_loaders[client_id], device=device, eta_loc=eta_loc)
        # Сохраняем обновленную локальную модель до следующего раунда
        local_models[client_id] = updated_local
        client_gradients.append(pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты и делает шаг
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

In [ ]:
#========================================================
# 9.2.2 Реализация алгоритма pFedSAM.
# Модель ResNet-18 для датасета CIFAR-10
#========================================================
print("Загрузка данных CIFAR-10...")
# Трансформации для CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов 

eta_loc = 0.1  # Стандартный шаг для локального обучения ResNet
eta_glob = 1.0  # Шаг сервера

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициализация ResNet-18 для CIFAR-10 на GPU
global_model = get_resnet18_cifar10().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA (ResNet-18 на CIFAR-10): Клиентов={num_clients}, Раундов={rounds}")

# Инициализируем локальные модели для каждого клиента
local_models = [copy.deepcopy(global_model).to(device) for _ in range(num_clients)]

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # Передаем глобальную модель и ПЕРСОНАЛЬНУЮ модель клиента
        pseudo_grad, updated_local = client_update_pfedsam(global_model, local_models[client_id], client_loaders[client_id], device=device, eta_loc=eta_loc)
        # Сохраняем обновленную локальную модель до следующего раунда
        local_models[client_id] = updated_local
        client_gradients.append(pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

# 9.3 Реализация Fed-NGA

In [ ]:
def server_aggregate_fednga(global_model, client_gradients, eta_glob):
    """
    Агрегация Fed-NGA: L2-нормализация псевдо-градиентов перед усреднением.
    Надежно защищает от атак масштабирования и сдвига знака (Sign-Flipping).
    """
    normalized_clients = []
    
    # --- Шаг 1: L2-нормализация вектора каждого клиента ---
    for client_grad in client_gradients:
        # Вычисляем общую L2-норму (длину) вектора всех слоев клиента
        sq_norm = 0.0
        for param_grad in client_grad:
            sq_norm += torch.sum(param_grad ** 2).item()
        
        # Добавляем 1e-12 для защиты от деления на ноль, если клиент прислал нулевой вектор
        l2_norm = (sq_norm ** 0.5) + 1e-12  
        
        # Создаем новый нормализованный вектор: направление сохраняется, длина = 1
        norm_grad = [param_grad / l2_norm for param_grad in client_grad]
        normalized_clients.append(norm_grad)
        
    # --- Шаг 2: Усреднение направлений и обновление глобальной модели ---
    num_clients = len(normalized_clients)
    
    with torch.no_grad():
        for param_idx, param_glob in enumerate(global_model.parameters()):
            # Создаем пустой тензор для накопления суммы направлений
            aggregated_direction = torch.zeros_like(param_glob.data)
            
            # Суммируем нормализованные векторы от всех клиентов
            for client_idx in range(num_clients):
                aggregated_direction.add_(normalized_clients[client_idx][param_idx])
                
            # Усредняем направление
            aggregated_direction.div_(num_clients)
            
            # Делаем глобальный шаг в усредненном направлении
            param_glob.data.sub_(eta_glob * aggregated_direction)

def client_update_standard(global_model, train_loader, device, eta_loc, epochs=1):
    """
    Стандартное локальное обучение (Local SGD).
    Используется как база для FedAvg, Fed-NGA и FedLAW.
    Возвращает только вектор смещения (псевдо-градиент).
    """
    # Создаем локальную копию модели для обучения
    local_model = copy.deepcopy(global_model).to(device)
    local_model.train()
    
    # Обычный базовый оптимизатор
    optimizer = torch.optim.SGD(local_model.parameters(), lr=eta_loc)
    loss_fn = torch.nn.CrossEntropyLoss()
    
    # Тренируем заданное количество локальных эпох (обычно 1)
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = local_model(data)
            loss = loss_fn(output, target)
            loss.backward()
            optimizer.step()
            
    # Вычисляем псевдо-градиент (разницу весов для сервера)
    client_pseudo_grad = []
    with torch.no_grad():
        for param_glob, param_loc in zip(global_model.parameters(), local_model.parameters()):
            # Разница: Глобальная - Локальная (направление спуска)
            client_pseudo_grad.append(param_glob.data - param_loc.data)
            
    return client_pseudo_grad

In [ ]:
#========================================================
# 9.3.1 Реализация алгоритма Fed-NGA.
# Модель CNN для датасета MNIST
#========================================================
print("Загрузка данных MNIST...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов
eta_loc = 0.1
eta_glob = 1.0

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициируем нашу модель
global_model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA: Клиентов={num_clients}, Раундов={rounds}")

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
            # Здесь вызывается твоя стандартная функция локального обучения
            pseudo_grad = client_update_standard(global_model, client_loaders[client_id], device=device, eta_loc=eta_loc)
            client_gradients.append(pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Обрати внимание: eta_glob здесь должен быть больше (например, 10.0 или 20.0), 
    # так как мы искусственно уменьшили векторы клиентов до длины 1.
    server_aggregate_fednga(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

In [ ]:
#========================================================
# 9.3.2 Реализация алгоритма Fed-NGA.
# Модель ResNet-18 для датасета CIFAR-10
#========================================================
print("Загрузка данных CIFAR-10...")
# Трансформации для CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов 

eta_loc = 0.1  # Стандартный шаг для локального обучения ResNet
eta_glob = 1.0  # Шаг сервера

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициализация ResNet-18 для CIFAR-10 на GPU
global_model = get_resnet18_cifar10().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA (ResNet-18 на CIFAR-10): Клиентов={num_clients}, Раундов={rounds}")

# Инициализируем локальные модели для каждого клиента
local_models = [copy.deepcopy(global_model).to(device) for _ in range(num_clients)]

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # Передаем глобальную модель и ПЕРСОНАЛЬНУЮ модель клиента
        pseudo_grad, updated_local = client_update_pfedsam(global_model, local_models[client_id], client_loaders[client_id], device=device, eta_loc=eta_loc)
        # Сохраняем обновленную локальную модель до следующего раунда
        local_models[client_id] = updated_local
        client_gradients.append(pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

# Реализация FedLAW

In [ ]:
def server_aggregate_fedlaw(global_model, client_gradients, prev_global_grad=None):
    """
    Агрегация FedLAW: Адаптивное взвешивание на основе сходства градиентов.
    prev_global_grad: вектор обновления всей модели на прошлом раунде (как эталон).
    """
    num_clients = len(client_gradients)
    
    # 1. Превращаем списки тензоров (слои) в один длинный плоский вектор для каждого клиента
    flat_client_grads = []
    for grads in client_gradients:
        flat_client_grads.append(torch.cat([g.flatten() for g in grads]))
    
    # 2. Если это первый раунд, эталона нет — используем простое среднее
    if prev_global_grad is None:
        weights = torch.ones(num_clients) / num_clients
    else:
        # Считаем косинусное сходство каждого клиента с эталоном (прошлым шагом системы)
        similarities = []
        flat_prev = torch.cat([g.flatten() for g in prev_global_grad])
        
        for flat_grad in flat_client_grads:
            sim = F.cosine_similarity(flat_grad.unsqueeze(0), flat_prev.unsqueeze(0))
            # Используем ReLU, чтобы отсечь (обнулить вес) тех, кто тянет в обратную сторону (Sign-Flip)
            similarities.append(torch.clamp(sim, min=0.0).item())
        
        # Нормализуем сходства, чтобы получить веса (сумма весов = 1)
        sim_sum = sum(similarities) + 1e-12
        weights = torch.tensor([s / sim_sum for s in similarities])

    # 3. Агрегация с вычисленными весами
    with torch.no_grad():
        for param_idx, param in enumerate(global_model.parameters()):
            aggregated_update = torch.zeros_like(param.data)
            for client_idx in range(num_clients):
                aggregated_update.add_(weights[client_idx] * client_gradients[client_idx][param_idx])
            
            # Обновляем модель (eta_glob здесь обычно 1.0)
            param.data.sub_(aggregated_update)
            
    return client_gradients # Возвращаем текущие градиенты, чтобы они стали "предыдущими" в следующем раунде

In [ ]:
#========================================================
# 9.4.1 Реализация алгоритма FedLAW.
# Модель CNN для датасета MNIST
#========================================================
print("Загрузка данных MNIST...")
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов
eta_loc = 0.1
eta_glob = 1.0

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициируем нашу модель
global_model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA: Клиентов={num_clients}, Раундов={rounds}")

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
            # Здесь вызывается твоя стандартная функция локального обучения
            pseudo_grad = client_update_standard(global_model, client_loaders[client_id], device=device, eta_loc=eta_loc)
            client_gradients.append(pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Обрати внимание: eta_glob здесь должен быть больше (например, 10.0 или 20.0), 
    # так как мы искусственно уменьшили векторы клиентов до длины 1.
    server_aggregate_fednga(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения

In [ ]:
#========================================================
# 9.4.2 Реализация алгоритма FedLAW.
# Модель ResNet-18 для датасета CIFAR-10
#========================================================
print("Загрузка данных CIFAR-10...")
# Трансформации для CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Настройки симуляции
num_clients = 8
rounds = 100
batch_size = 64
num_attackers = 1  # 1 хакер из 8 клиентов 

eta_loc = 0.1  # Стандартный шаг для локального обучения ResNet
eta_glob = 1.0  # Шаг сервера

# Тут решаем, будет ли у нас всё хорошо, или же суровая реальность с Non-IDD
# === Распределяем данные поровну между клиентами (IID распределение для теста) --- хорошо ===
#data_per_client = len(train_dataset) // num_clients
#client_loaders = []
#for i in range(num_clients):
#    indices = list(range(i * data_per_client, (i + 1) * data_per_client))
#    subset = Subset(train_dataset, indices)
#    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
#    # Сохраняем итератор для удобного получения батчей
#    client_loaders.append(iter(loader))
# ===========================================================================================

# === Non-IID Распределение данных === (раскомментировать строки внизу и закомментировать сверху)
beta_value = 100 # Измени на 100 для IID, или на 0.1 для экстремального перекоса
client_datasets = partition_data_dirichlet(train_dataset, num_clients, beta=beta_value)

client_loaders = []
for subset in client_datasets:
    loader = DataLoader(subset, batch_size=batch_size, shuffle=True)
    # Сохраняем итератор для удобного получения батчей
    client_loaders.append(loader)
# =============================================

# Инициализация ResNet-18 для CIFAR-10 на GPU
global_model = get_resnet18_cifar10().to(device)
loss_fn = nn.CrossEntropyLoss()

print(f"Запуск Fed-SANA (ResNet-18 на CIFAR-10): Клиентов={num_clients}, Раундов={rounds}")

# Инициализируем локальные модели для каждого клиента
local_models = [copy.deepcopy(global_model).to(device) for _ in range(num_clients)]

for round_idx in range(rounds):
    client_gradients = []

    # 1. Фаза клиентов
    for client_id in range(num_clients):
        # Передаем глобальную модель и ПЕРСОНАЛЬНУЮ модель клиента
        pseudo_grad, updated_local = client_update_pfedsam(global_model, local_models[client_id], client_loaders[client_id], device=device, eta_loc=eta_loc)
        # Сохраняем обновленную локальную модель до следующего раунда
        local_models[client_id] = updated_local
        client_gradients.append(pseudo_grad)

    # === Внедрение византийской атаки ===
    # Выбрать любую атаку:
    #client_gradients = attack_sign_flipping(client_gradients, num_attackers, gamma=10.0)
    #client_gradients = attack_a_little_is_enough(client_gradients, num_attackers, z=1.5)
    # ============================================

    # 2. Фаза сервера
    # Сервер агрегирует нормализованные градиенты
    server_aggregate_and_update(global_model, client_gradients, eta_glob=eta_glob)

    # 3. Оценка глобальной модели (каждые 2 раунда)
    if round_idx % 2 == 0 or round_idx == rounds - 1:
        accuracy, f1, optimality_gap = evaluate_metrics(global_model, test_loader, loss_fn, device)

        print(f"Раунд {round_idx+1:03d}/{rounds} | Accuracy: {accuracy:.2f}% | F1-Score: {f1:.2f}% | Opt. Gap: {optimality_gap:.4f} | Time: {datetime.now().strftime('%H:%M:%S')}")

        global_model.train() # Возвращаем модель в режим обучения